In [1]:
from pathlib import Path

results_dir = Path(
    r"C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase09_2_overfitting_v2"
)

print("Folder exists:", results_dir.exists())

if results_dir.exists():
    print("\nCSV files:")
    for path in results_dir.rglob("*.csv"):
        print(path.relative_to(results_dir))

    print("\nCheckpoint files:")
    for path in results_dir.rglob("*.pt"):
        print(path.relative_to(results_dir))

    print("\nJSON status files:")
    for path in results_dir.rglob("*.json"):
        print(path.relative_to(results_dir))

Folder exists: True

CSV files:
phase09_2_overfitting_summary.csv
phase09_2_vs_control.csv
regularization\vgg16\cric\history.csv
regularization\vgg16\riva\history.csv
regularization\swin_tiny\cric\history.csv
regularization\swin_tiny\riva\history.csv
regularization\convnext_tiny\cric\history.csv
regularization\convnext_tiny\riva\history.csv
control\vgg16\cric\history.csv
control\vgg16\riva\history.csv
control\swin_tiny\cric\history.csv
control\swin_tiny\riva\history.csv
control\convnext_tiny\cric\history.csv
control\convnext_tiny\riva\history.csv
combined\vgg16\cric\history.csv
combined\vgg16\riva\history.csv
combined\swin_tiny\cric\history.csv
combined\swin_tiny\riva\history.csv
combined\convnext_tiny\cric\history.csv
combined\convnext_tiny\riva\history.csv
augmentation\vgg16\cric\history.csv
augmentation\vgg16\riva\history.csv
augmentation\swin_tiny\cric\history.csv
augmentation\swin_tiny\riva\history.csv
augmentation\convnext_tiny\cric\history.csv
augmentation\convnext_tiny\riva\his

In [2]:
from pathlib import Path
import json
import pandas as pd

# Change this only if your project is in a different location
PROJECT_ROOT = Path(r"C:\Users\nanda\Documents\cervical-semi-sl")

print("=" * 80)
print("PHASE 09 RESULTS INSPECTION — TEXT ONLY")
print("=" * 80)
print("Project root exists:", PROJECT_ROOT.exists())
print("Project root:", PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}\n"
        "Update PROJECT_ROOT to your actual project path."
    )

# Search result files and metric JSON files; exclude large raw data folders
SKIP_DIRS = {
    ".git", "__pycache__", "cric_cervix", "riva_1.0",
    "images", "checkpoints", "venv", ".venv"
}

matches = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if any(part.lower() in SKIP_DIRS for part in path.parts):
        continue

    suffix = path.suffix.lower()
    name = path.name.lower()

    if suffix == ".csv" and any(
        word in name for word in ["result", "summary", "metric", "benchmark", "test"]
    ):
        matches.append(path)

    elif suffix == ".json" and any(
        word in name for word in ["metric", "result", "summary", "test", "validation"]
    ):
        matches.append(path)

# Prefer files that appear related to the original Phase 09 benchmark
matches = sorted(
    set(matches),
    key=lambda p: (
        0 if "phase09" in str(p).lower() or "phase_09" in str(p).lower() else 1,
        str(p).lower()
    )
)

print(f"\nFound {len(matches)} possible result/metric files.")

for i, path in enumerate(matches, start=1):
    print(f"{i:03d}. {path.relative_to(PROJECT_ROOT)}")

print("\n" + "=" * 80)
print("CONTENTS OF RELEVANT RESULT FILES")
print("=" * 80)

# Print compact CSV tables and JSON metric/config summaries
MAX_FILES = 30
MAX_ROWS = 40
MAX_COLUMNS = 35

for path in matches[:MAX_FILES]:
    print("\n" + "-" * 80)
    print("FILE:", path.relative_to(PROJECT_ROOT))
    print("-" * 80)

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path)

            # Show columns first so we can identify split/model/metric fields
            print("Columns:", list(df.columns))
            print("Rows:", len(df))

            # Prioritize RIVA rows where a dataset column exists
            dataset_cols = [
                c for c in df.columns
                if any(k in c.lower() for k in ["dataset", "domain", "source", "target"])
            ]

            if dataset_cols:
                mask = pd.Series(False, index=df.index)
                for col in dataset_cols:
                    mask |= df[col].astype(str).str.contains(
                        "riva", case=False, na=False
                    )
                if mask.any():
                    print("\nRIVA-related rows:")
                    print(df.loc[mask].iloc[:MAX_ROWS, :MAX_COLUMNS].to_string(index=False))
                else:
                    print("\nNo rows explicitly mention RIVA. Showing first rows:")
                    print(df.iloc[:MAX_ROWS, :MAX_COLUMNS].to_string(index=False))
            else:
                print(df.iloc[:MAX_ROWS, :MAX_COLUMNS].to_string(index=False))

        elif path.suffix.lower() == ".json":
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Print metric-related JSON fields; retain nested metric dictionaries
            print(json.dumps(data, indent=2, ensure_ascii=False)[:12000])

    except Exception as exc:
        print("Could not read file:", repr(exc))

if len(matches) > MAX_FILES:
    print(
        f"\nOnly the first {MAX_FILES} files were printed. "
        f"There are {len(matches) - MAX_FILES} additional matches."
    )

print("\n" + "=" * 80)
print("INSPECTION COMPLETE — NO TRAINING OR IMAGE DISPLAY PERFORMED")
print("=" * 80)

PHASE 09 RESULTS INSPECTION — TEXT ONLY
Project root exists: True
Project root: C:\Users\nanda\Documents\cervical-semi-sl

Found 83 possible result/metric files.
001. data\results\phase09_2_overfitting_v2\augmentation\convnext_tiny\cric\best_validation_metrics.json
002. data\results\phase09_2_overfitting_v2\augmentation\convnext_tiny\riva\best_validation_metrics.json
003. data\results\phase09_2_overfitting_v2\augmentation\swin_tiny\cric\best_validation_metrics.json
004. data\results\phase09_2_overfitting_v2\augmentation\swin_tiny\riva\best_validation_metrics.json
005. data\results\phase09_2_overfitting_v2\augmentation\vgg16\cric\best_validation_metrics.json
006. data\results\phase09_2_overfitting_v2\augmentation\vgg16\riva\best_validation_metrics.json
007. data\results\phase09_2_overfitting_v2\combined\convnext_tiny\cric\best_validation_metrics.json
008. data\results\phase09_2_overfitting_v2\combined\convnext_tiny\riva\best_validation_metrics.json
009. data\results\phase09_2_overfittin

In [3]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\nanda\Documents\cervical-semi-sl")
RESULTS_DIR = PROJECT_ROOT / "data" / "results" / "phase09_v2"

print("=" * 90)
print("PHASE 09 — FULL SUPERVISED TEST METRICS")
print("=" * 90)

# 1. Print the exact saved test metrics for every model and dataset
for dataset in ["riva", "cric"]:
    print(f"\n{'=' * 35} {dataset.upper()} {'=' * 35}")

    for model_dir in sorted(RESULTS_DIR.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        dataset_dir = model_dir / dataset
        metrics_file = dataset_dir / "test_metrics.json"
        per_class_file = dataset_dir / "per_class_metrics.csv"

        if not metrics_file.exists():
            continue

        print(f"\n--- Model: {model_name} ---")

        try:
            with open(metrics_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            print("Test metrics:")
            print(json.dumps(data, indent=2))

        except Exception as e:
            print("Error reading test metrics:", repr(e))

        if per_class_file.exists():
            try:
                print("\nPer-class metrics:")
                print(pd.read_csv(per_class_file).to_string(index=False))
            except Exception as e:
                print("Error reading per-class metrics:", repr(e))


# 2. Print the original Phase 09 model-selection summary
summary_file = RESULTS_DIR / "phase09_v2_summary.csv"

print("\n" + "=" * 90)
print("PHASE 09 SUMMARY CSV")
print("=" * 90)

if summary_file.exists():
    summary = pd.read_csv(summary_file)
    print("Columns:", list(summary.columns))
    print(summary.to_string(index=False))
else:
    print("Summary CSV not found:", summary_file)


# 3. Print the training diagnostics summary, if available
diagnostics_file = RESULTS_DIR / "training_diagnostics_summary_v2.csv"

print("\n" + "=" * 90)
print("TRAINING DIAGNOSTICS SUMMARY")
print("=" * 90)

if diagnostics_file.exists():
    diagnostics = pd.read_csv(diagnostics_file)
    print("Columns:", list(diagnostics.columns))
    print(diagnostics.to_string(index=False))
else:
    print("Diagnostics summary CSV not found:", diagnostics_file)

print("\nInspection finished. No training performed; no images displayed.")

PHASE 09 — FULL SUPERVISED TEST METRICS

=================================== RIVA ===================================

--- Model: convnext_tiny ---
Test metrics:
{
  "dataset": "riva",
  "model": "convnext_tiny",
  "checkpoint": "checkpoint_best.pt",
  "evaluation_split": "test",
  "class_mapping": {
    "0": "Normal / Non-pathological",
    "1": "Abnormal / Pathological"
  },
  "metrics": {
    "accuracy": 0.8250938673341677,
    "precision": 0.7565610859728507,
    "recall_sensitivity": 0.7424511545293073,
    "specificity": 0.870048309178744,
    "macro_f1": 0.8075485035264275,
    "auroc": 0.8942642503496624,
    "tn": 1801.0,
    "fp": 269.0,
    "fn": 290.0,
    "tp": 836.0
  }
}

Per-class metrics:
                    class  precision  recall_sensitivity  f1_score  support
Normal / Non-pathological   0.861310            0.870048  0.865657     2070
  Abnormal / Pathological   0.756561            0.742451  0.749440     1126

--- Model: densenet121 ---
Test metrics:
{
  "dataset": 